# Pilot study (Section 4.1.3): classical ML baselines on GAIA — FAIR comparison

**This is the corrected version of the pilot study notebook.** The previous version loaded pre-computed embeddings from `data/gaia/tmp/*.pkl`, which were produced by a single global FastText training pass (either on the original 15/85 train split, or on a single TransTVDiag CV fold). That made every classifier evaluation suffer from preprocessing leakage: FastText word embeddings were fit to maximize separability of `__label__{node_idx}{type_idx}` markers, and the sklearn CV then tested on rows whose labels FastText had already seen during its supervised training. This produced artificially inflated scores that made simple classifiers appear to beat TransTVDiag.

The fair protocol used here matches what `main.py` does for TransTVDiag CV ([main.py:211-277](../../TransTVDiag_attention_analysis/main.py)):

1. Split indices with `StratifiedKFold` (same `random_state` / `n_splits` / stratification column as `main.py`).
2. For each fold, retrain FastText from scratch using only that fold's `train_indices`.
3. Compute embeddings for **all** rows using the just-trained FastText, then split into train/test by fold indices.
4. Fit sklearn classifiers on the train slice, score on the test slice.

Two feature variants are still evaluated, but now both are reproducible end-to-end:

| Variant            | Dim   | Notes |
|--------------------|-------|-------|
| `full_graph`       | 3840  | Honest baseline — same fold-aware preprocessing as TransTVDiag |
| `root_cause_only`  | 384   | Diagnostic — picks the root-cause node's embedding. FastText still uses `__label__{node}{type}` markers, so this is the *strongest* signal classical ML can extract from the per-fold embeddings |

Six classifiers, same family as in Section 4.1.1 plus LightGBM.

## Setup

In [1]:
from __future__ import annotations
from pathlib import Path
import json
import pickle
import time
import warnings

import numpy as np
import pandas as pd


In [8]:
!pip install scikit-learn
!pip install lightgbm
!pip install fasttext

Looking in indexes: https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/simple

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Looking in indexes: https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/simple

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Looking in indexes: https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 8.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/packages/packages/pybind11/3.0.4/pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
Using cached https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/packages/packages/pybind11/3.

In [3]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, ndcg_score
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

try:
    from lightgbm import LGBMClassifier
    HAVE_LGBM = True
except ImportError:
    HAVE_LGBM = False

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', message='X does not have valid feature names')
print('LightGBM available:', HAVE_LGBM)


LightGBM available: True


In [9]:
# Repo root must be on sys.path so we can reuse FastTextEncoder from the
# TransTVDiag codebase — that guarantees we use the exact same encoder as the
# main model and have nothing to drift out of sync.
import sys
REPO_ROOT = Path('../')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from process.events.fasttext_w2v import FastTextEncoder
from helper import io_util


In [10]:
# Paths
DATA_DIR = REPO_ROOT / 'data' / 'gaia'
EVENTS_DIR = DATA_DIR / 'events'
LABELS_CSV = DATA_DIR / 'gaia.csv'
OUT_DIR = Path('../figures')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Match main.py CV defaults so test/train splits are aligned with TransTVDiag.
N_FOLDS = 5
SEED = 42                          # main.py default for CV runs in train.sh
CV_STRATIFY_BY = 'instance'        # same as thesis Section 4.2 protocol
EMBEDDING_DIM = 128                # main.py default --embedding_dim
FASTTEXT_EPOCHS = 5                # EventProcess.build_embedding default


In [11]:
# Canonical class orders
RCL_ORDER = [
    'mobservice1', 'mobservice2',
    'webservice1', 'webservice2',
    'dbservice1', 'dbservice2',
    'logservice1', 'logservice2',
    'redisservice1', 'redisservice2',
]

FTI_SHORT = {
    '[access permission denied exception]': 'access_perm',
    '[file moving program]':                'file_moving',
    '[login failure]':                      'login_failure',
    '[memory_anomalies]':                   'memory_anomalies',
    '[normal memory freed label]':          'normal_mem_freed',
}
FTI_TYPES = ['access_perm', 'file_moving', 'login_failure',
             'memory_anomalies', 'normal_mem_freed']

RCL_MINORITY = ['webservice1', 'webservice2', 'dbservice1', 'dbservice2',
                'logservice1', 'logservice2', 'redisservice1', 'redisservice2']


## Load raw event data and labels

Read once at the top, reuse across folds. Mirrors `EventProcess.process` ([EventProcess.py:18-44](../../TransTVDiag_attention_analysis/process/EventProcess.py)).

In [12]:
with open(EVENTS_DIR / 'metric.json', 'r', encoding='utf8') as fp:
    metrics_raw = json.load(fp)
with open(EVENTS_DIR / 'trace.json', 'r', encoding='utf8') as fp:
    traces_raw = json.load(fp)
with open(EVENTS_DIR / 'log.json', 'r', encoding='utf8') as fp:
    logs_raw = json.load(fp)

edges = io_util.load(str(EVENTS_DIR / 'edges.pkl'))
nodes = io_util.load(str(EVENTS_DIR / 'nodes.pkl'))

print(f'nodes ({len(nodes)}): {nodes}')
print(f'edges: {len(edges[0])} directed edges')
print(f'event JSONs loaded: metric={len(metrics_raw)} trace={len(traces_raw)} log={len(logs_raw)}')


nodes (10): ['dbservice1', 'dbservice2', 'logservice1', 'logservice2', 'mobservice1', 'mobservice2', 'redisservice1', 'redisservice2', 'webservice1', 'webservice2']
edges: 26 directed edges
event JSONs loaded: metric=16205 trace=16205 log=16205


In [13]:
labels_df = pd.read_csv(LABELS_CSV)
labels_df = (labels_df[['index', 'anomaly_type', 'instance', 'data_type']]
             .drop_duplicates('index')
             .sort_values('index')
             .reset_index(drop=True))
labels_df['fti'] = labels_df['anomaly_type'].map(FTI_SHORT).fillna(labels_df['anomaly_type'])

# Match EventProcess.process line 39: types = ['normal'] + unique anomaly types
types_for_fasttext = ['normal'] + labels_df['anomaly_type'].unique().tolist()

print(f'rows in gaia.csv: {len(labels_df)}')
print(f'types_for_fasttext: {types_for_fasttext}')
labels_df.head()


rows in gaia.csv: 16205
types_for_fasttext: ['normal', '[memory_anomalies]', '[normal memory freed label]', '[login failure]', '[file moving program]', '[access permission denied exception]']


,index,anomaly_type,instance,data_type,fti
0,0,[memory_anomalies],dbservice1,train,memory_anomalies
1,1,[normal memory freed label],dbservice1,train,normal_mem_freed
2,2,[memory_anomalies],dbservice1,train,memory_anomalies
3,3,[memory_anomalies],dbservice2,train,memory_anomalies
4,4,[memory_anomalies],dbservice2,train,memory_anomalies


## Per-fold FastText embedder

Replicates `EventProcess.build_embedding(train_indices=...)` ([EventProcess.py:46-97](../../TransTVDiag_attention_analysis/process/EventProcess.py)) but in-memory: trains one FastText encoder per modality on `train_indices`, then computes embeddings for all rows and returns the three (N, 10, 128) tensors without ever touching `data/gaia/tmp/`.

In [14]:
def _build_docs_for_modality(modality, raw_data, idx_list, nodes):
    '''Reproduces the doc construction from EventProcess.build_embedding.
    `idx_list` is a sequence of row indices into raw_data. For each row,
    produces one doc per node (10 docs per row).'''
    docs = []
    for idx in idx_list:
        events = raw_data[str(idx)]
        for node in nodes:
            if modality == 'trace':
                doc = ['&'.join(e) for e in events
                       if (node in e[0] or node in e[1])]
            else:
                doc = ['&'.join(e) for e in events if node in e[0]]
            docs.append(doc)
    return docs


In [15]:
def _build_supervised_labels(train_rows, nodes, types_list):
    '''For each (row, node) pair: __label__{node_idx}{type_idx} if that node
    is the root cause, otherwise __label__{node_idx}0. Same scheme as
    EventProcess.build_embedding lines 75-80.'''
    labels = []
    ins_col = train_rows['instance'].values
    type_col = train_rows['anomaly_type'].values
    for i in range(len(train_rows)):
        for node in nodes:
            if node == ins_col[i]:
                labels.append(f'__label__{nodes.index(node)}'
                              f'{types_list.index(type_col[i])}')
            else:
                labels.append(f'__label__{nodes.index(node)}0')
    return labels


In [16]:
def compute_fold_embeddings(train_indices, labels_df, raw_streams,
                            nodes, types_list,
                            embedding_dim=EMBEDDING_DIM,
                            epochs=FASTTEXT_EPOCHS):
    '''Train one FastText per modality on `train_indices`, return (N, 10, 128)
    tensors for metric/trace/log, where N = len(labels_df).

    raw_streams: dict {'metric': metrics_raw, 'trace': traces_raw, 'log': logs_raw}
    '''
    train_rows = labels_df.iloc[np.asarray(train_indices)]
    train_row_indices = train_rows['index'].values.tolist()
    all_row_indices = labels_df['index'].values.tolist()
    N = len(labels_df)
    n_nodes = len(nodes)

    sup_labels = _build_supervised_labels(train_rows, nodes, types_list)

    out = {}
    for key, data in raw_streams.items():
        encoder = FastTextEncoder(
            key, nodes, types_list,
            embedding_dim=embedding_dim, epochs=epochs,
        )
        train_docs = _build_docs_for_modality(key, data, train_row_indices, nodes)
        encoder.fit(train_docs, sup_labels)

        # Encode all rows (including test) with the just-trained encoder.
        embs = np.zeros((N, n_nodes, embedding_dim), dtype=np.float32)
        for i, idx in enumerate(all_row_indices):
            for j, node in enumerate(nodes):
                if key == 'trace':
                    doc = ['&'.join(e) for e in data[str(idx)]
                           if (node in e[0] or node in e[1])]
                else:
                    doc = ['&'.join(e) for e in data[str(idx)]
                           if node in e[0]]
                embs[i, j] = encoder.get_sentence_embedding(doc)
        out[key] = embs
    return out['metric'], out['trace'], out['log']


## Feature variants and metric helpers

In [17]:
def build_full_graph(metric, trace, log):
    x = np.concatenate([metric, trace, log], axis=2)
    return x.reshape(x.shape[0], -1).astype(np.float32)

def build_root_cause_only(metric, trace, log, labels):
    rc_idx = labels['instance'].map(lambda s: RCL_ORDER.index(s)).values
    rows = np.arange(metric.shape[0])
    return np.concatenate(
        [metric[rows, rc_idx], trace[rows, rc_idx], log[rows, rc_idx]],
        axis=1,
    ).astype(np.float32)


In [18]:
def hr_at_k(probs, y_true_idx, k):
    top_k = np.argsort(-probs, axis=1)[:, :k]
    return float((top_k == y_true_idx[:, None]).any(axis=1).mean())

def ndcg_at_3(probs, y_true_idx):
    n, c = probs.shape
    y_rel = np.zeros((n, c), dtype=np.float32)
    y_rel[np.arange(n), y_true_idx] = 1.0
    return float(ndcg_score(y_rel, probs, k=3))

def align_proba(clf, classes_in_order):
    seen = list(clf.classes_)
    return np.asarray([seen.index(c) if c in seen else -1
                       for c in classes_in_order])


## Classifier zoo

In [19]:
def build_classifiers():
    clfs = {
        'GaussianNB':   GaussianNB(),
        'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=0),
        'KNN':          KNeighborsClassifier(n_neighbors=5),
        'RandomForest': RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            n_jobs=-1, random_state=0,
        ),
        'LogReg':       LogisticRegression(
            max_iter=2000, class_weight='balanced', n_jobs=-1,
        ),
    }
    if HAVE_LGBM:
        clfs['LightGBM'] = LGBMClassifier(
            n_estimators=200, class_weight='balanced',
            n_jobs=-1, random_state=0, verbose=-1,
        )
    else:
        clfs['GradientBoosting'] = GradientBoostingClassifier(
            n_estimators=200, random_state=0,
        )
    return clfs

list(build_classifiers().keys())


['GaussianNB', 'DecisionTree', 'KNN', 'RandomForest', 'LogReg', 'LightGBM']

## Per-fold evaluator and CV runner

Key change vs the previous notebook: `run_cv` now calls `compute_fold_embeddings` inside the fold loop, so FastText is retrained from scratch on every fold's training indices. After per-fold embeddings are computed, both `full_graph` and `root_cause_only` features for both FTI and RCL are evaluated in a single pass — this amortises the FastText cost (the expensive step).

In [20]:
def evaluate_one_fold(clf, X_tr, X_te, y_tr, y_te, task, classes):
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)

    out = {}
    out['macro_f1'] = f1_score(y_te, y_pred, labels=classes,
                               average='macro', zero_division=0)
    for cls, f1 in zip(classes,
                       f1_score(y_te, y_pred, labels=classes,
                                average=None, zero_division=0)):
        out[f'f1_{cls}'] = float(f1)

    if task == 'RCL' and hasattr(clf, 'predict_proba'):
        raw = clf.predict_proba(X_te)
        perm = align_proba(clf, classes)
        probs = np.zeros((raw.shape[0], len(classes)), dtype=np.float32)
        for j, p in enumerate(perm):
            if p >= 0:
                probs[:, j] = raw[:, p]
        y_te_idx = np.array([classes.index(y) for y in y_te])
        out['hr_1'] = hr_at_k(probs, y_te_idx, 1)
        out['hr_3'] = hr_at_k(probs, y_te_idx, 3)
        out['ndcg_3'] = ndcg_at_3(probs, y_te_idx)
    return out


In [21]:
def run_cv_per_fold_fasttext(labels_df, raw_streams, nodes, types_list,
                             n_folds=N_FOLDS, seed=SEED):
    '''Outer 5-fold stratified CV. For each fold:
       1. Retrain FastText on train_indices only.
       2. Build full_graph and root_cause_only features.
       3. Train each classifier on the train slice, score on the test slice.
       4. Record metrics for both feature variants × both tasks × all classifiers.
    Returns: results[variant][task][classifier] -> list of per-fold metric dicts.'''
    y_fti = labels_df['fti'].values
    y_rcl = labels_df['instance'].values
    stratify = labels_df[CV_STRATIFY_BY].astype(str).values

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    clf_names = list(build_classifiers().keys())
    results = {
        'full_graph':      {'FTI': {}, 'RCL': {}},
        'root_cause_only': {'FTI': {}, 'RCL': {}},
    }

    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(labels_df, stratify)):
        print(f'\n[fold {fold_idx + 1}/{n_folds}] training FastText on'
              f' {len(tr_idx)} rows...', flush=True)
        t_emb = time.time()
        m_emb, t_emb_arr, l_emb = compute_fold_embeddings(
            tr_idx, labels_df, raw_streams, nodes, types_list
        )
        print(f'  FastText + embedding done in {time.time() - t_emb:.1f}s', flush=True)

        X_full = build_full_graph(m_emb, t_emb_arr, l_emb)
        X_rc = build_root_cause_only(m_emb, t_emb_arr, l_emb, labels_df)

        for variant, X in [('full_graph', X_full), ('root_cause_only', X_rc)]:
            scaler = StandardScaler(with_mean=True, with_std=True)
            X_tr = scaler.fit_transform(X[tr_idx])
            X_te = scaler.transform(X[te_idx])

            for task, y, classes in [('FTI', y_fti, FTI_TYPES),
                                     ('RCL', y_rcl, RCL_ORDER)]:
                for clf_name in clf_names:
                    clf = build_classifiers()[clf_name]
                    t0 = time.time()
                    try:
                        m = evaluate_one_fold(clf, X_tr, X_te,
                                              y[tr_idx], y[te_idx],
                                              task, classes)
                        m['_time_sec'] = time.time() - t0
                    except Exception as e:
                        print(f'  [{variant}/{task}/{clf_name}] FAILED:'
                              f' {type(e).__name__}: {e}', flush=True)
                        m = {'_error': f'{type(e).__name__}: {e}'}
                    results[variant][task].setdefault(clf_name, []).append(m)
        print(f'  fold {fold_idx + 1} total time:'
              f' {time.time() - t_emb:.1f}s', flush=True)
    return results


## Run the full pipeline

Expect 5 × (FastText × 3 modalities + 6 classifiers × 2 tasks × 2 variants) iterations. FastText training dominates wall-clock; total runtime depends on the machine but is typically 20-60 minutes on a single CPU.

In [22]:
raw_streams = {'metric': metrics_raw, 'trace': traces_raw, 'log': logs_raw}

t_start = time.time()
results = run_cv_per_fold_fasttext(
    labels_df, raw_streams, nodes, types_for_fasttext,
    n_folds=N_FOLDS, seed=SEED,
)
print(f'\nTotal wall-clock: {(time.time() - t_start) / 60:.1f} min')



[fold 1/5] training FastText on 12964 rows...


Read 1M words
Number of words:  1023
Number of labels: 34
Progress: 100.0% words/sec/thread:   32595 lr:  0.000000 avg.loss:  1.446028 ETA:   0h 0m 0s
Read 2M words
Number of words:  1023
Number of labels: 34
Progress: 100.0% words/sec/thread:   42835 lr:  0.000000 avg.loss:  0.985597 ETA:   0h 0m 0s
Read 0M words
Number of words:  45
Number of labels: 34
Progress: 100.0% words/sec/thread:   12060 lr:  0.000000 avg.loss:  0.527929 ETA:   0h 0m 0s
Read 0M words
Number of words:  45
Number of labels: 34
Progress: 100.0% words/sec/thread:   14754 lr:  0.000000 avg.loss:  0.625496 ETA:   0h 0m 0s
Read 0M words
Number of words:  114
Number of labels: 34
Progress: 100.0% words/sec/thread:   16505 lr:  0.000000 avg.loss:  0.337333 ETA:   0h 0m 0s
Read 0M words
Number of words:  114
Number of labels: 34
Progress: 100.0% words/sec/thread:   17551 lr:  0.000000 avg.loss:  0.369377 ETA:   0h 0m 0s


  FastText + embedding done in 50.2s
  fold 1 total time: 670.1s

[fold 2/5] training FastText on 12964 rows...


Read 1M words
Number of words:  1023
Number of labels: 34
Progress: 100.0% words/sec/thread:   32069 lr:  0.000000 avg.loss:  1.475812 ETA:   0h 0m 0s
Read 2M words
Number of words:  1023
Number of labels: 34
Progress: 100.0% words/sec/thread:   44616 lr:  0.000000 avg.loss:  0.801601 ETA:   0h 0m 0s
Read 0M words
Number of words:  45
Number of labels: 34
Progress: 100.0% words/sec/thread:   12065 lr:  0.000000 avg.loss:  0.624494 ETA:   0h 0m 0s
Read 0M words
Number of words:  45
Number of labels: 34
Progress: 100.0% words/sec/thread:   14036 lr:  0.000000 avg.loss:  0.563603 ETA:   0h 0m 0s
Read 0M words
Number of words:  113
Number of labels: 34
Progress: 100.0% words/sec/thread:   16503 lr:  0.000000 avg.loss:  0.313576 ETA:   0h 0m 0s
Read 0M words
Number of words:  113
Number of labels: 34
Progress: 100.0% words/sec/thread:   17568 lr:  0.000000 avg.loss:  0.367912 ETA:   0h 0m 0s


  FastText + embedding done in 51.2s
  fold 2 total time: 722.5s

[fold 3/5] training FastText on 12964 rows...


...

  FastText + embedding done in 50.8s
  fold 3 total time: 696.5s

[fold 4/5] training FastText on 12964 rows...


Read 1M words
Number of words:  1023
Number of labels: 34
Progress: 100.0% words/sec/thread:   33396 lr:  0.000000 avg.loss:  0.952255 ETA:   0h 0m 0s
Read 2M words
Number of words:  1023
Number of labels: 34
Progress: 100.0% words/sec/thread:   44140 lr:  0.000000 avg.loss:  1.076074 ETA:   0h 0m 0s
Read 0M words
Number of words:  45
Number of labels: 34
Progress: 100.0% words/sec/thread:   12041 lr:  0.000000 avg.loss:  0.608916 ETA:   0h 0m 0s
Read 0M words
Number of words:  45
Number of labels: 34
Progress: 100.0% words/sec/thread:   14654 lr:  0.000000 avg.loss:  0.575857 ETA:   0h 0m 0s
Read 0M words
Number of words:  115
Number of labels: 34
Progress: 100.0% words/sec/thread:   16491 lr:  0.000000 avg.loss:  0.334234 ETA:   0h 0m 0s
Read 0M words
Number of words:  115
Number of labels: 34
Progress: 100.0% words/sec/thread:   17582 lr:  0.000000 avg.loss:  0.410909 ETA:   0h 0m 0s


  FastText + embedding done in 49.8s
  fold 4 total time: 687.6s

[fold 5/5] training FastText on 12964 rows...


Read 1M words
Number of words:  1023
Number of labels: 32
Progress: 100.0% words/sec/thread:   35135 lr:  0.000000 avg.loss:  0.909336 ETA:   0h 0m 0s
Read 2M words
Number of words:  1023
Number of labels: 32
Progress: 100.0% words/sec/thread:   39155 lr:  0.000000 avg.loss:  0.736462 ETA:   0h 0m 0s
Read 0M words
Number of words:  45
Number of labels: 32
Progress: 100.0% words/sec/thread:   13067 lr:  0.000000 avg.loss:  0.554743 ETA:   0h 0m 0s
Read 0M words
Number of words:  45
Number of labels: 32
Progress: 100.0% words/sec/thread:   15382 lr:  0.000000 avg.loss:  0.617195 ETA:   0h 0m 0s
Read 0M words
Number of words:  115
Number of labels: 32
Progress: 100.0% words/sec/thread:   17410 lr:  0.000000 avg.loss:  0.348215 ETA:   0h 0m 0s
Read 0M words
Number of words:  115
Number of labels: 32
Progress: 100.0% words/sec/thread:   18971 lr:  0.000000 avg.loss:  0.416958 ETA:   0h 0m 0s


  FastText + embedding done in 49.6s
  fold 5 total time: 690.7s

Total wall-clock: 57.8 min


## Aggregate per-fold metrics

In [23]:
def summarize(results):
    rows = []
    for variant, by_task in results.items():
        for task, by_clf in by_task.items():
            for clf_name, fold_metrics in by_clf.items():
                valid = [m for m in fold_metrics if '_error' not in m]
                if not valid:
                    rows.append({'variant': variant, 'task': task,
                                 'classifier': clf_name,
                                 '_error': 'all folds failed'})
                    continue
                row = {'variant': variant, 'task': task,
                       'classifier': clf_name,
                       'n_folds_ok': len(valid)}
                all_keys = set()
                for m in valid:
                    all_keys.update(m.keys())
                for k in sorted(all_keys):
                    vals = [m[k] for m in valid
                            if k in m and isinstance(m[k], (int, float))]
                    if not vals:
                        continue
                    row[f'{k}_mean'] = float(np.mean(vals))
                    row[f'{k}_std'] = float(np.std(vals))
                rows.append(row)
    return pd.DataFrame(rows)

final = summarize(results)
final.shape


(24, 44)

## Headline tables for the thesis

In [24]:
# FTI: macro-F1 + per-class for access_perm and other minority types
fti_view = final[final['task'] == 'FTI'][[
    'variant', 'classifier', 'macro_f1_mean', 'macro_f1_std',
    'f1_access_perm_mean', 'f1_file_moving_mean',
    'f1_memory_anomalies_mean', 'f1_normal_mem_freed_mean',
    'f1_login_failure_mean',
]].copy()
fti_view


,variant,classifier,macro_f1_mean,macro_f1_std,f1_access_perm_mean,f1_file_moving_mean,f1_memory_anomalies_mean,f1_normal_mem_freed_mean,f1_login_failure_mean
0,full_graph,GaussianNB,0.585623,0.070560,0.693333,0.380739,0.857672,0.000000,0.996370
1,full_graph,DecisionTree,0.727676,0.057642,0.845714,0.853968,0.939925,0.000000,0.998773
2,full_graph,KNN,0.643805,0.052359,0.724762,0.537335,0.957250,0.000000,0.999677
3,full_graph,RandomForest,0.745105,0.036336,0.766667,0.981818,0.977137,0.000000,0.999903
4,full_graph,LogReg,0.708433,0.050707,0.804762,0.799930,0.938574,0.000000,0.998901
5,full_graph,LightGBM,0.770121,0.025627,0.904762,0.965818,0.980122,0.000000,0.999903
12,root_cause_only,GaussianNB,0.451672,0.026830,0.316450,0.381400,0.573796,0.000000,0.986715
13,root_cause_only,DecisionTree,0.516982,0.052082,0.477460,0.218713,0.892103,0.000000,0.996632
14,root_cause_only,KNN,0.498597,0.055843,0.311111,0.259524,0.923450,0.000000,0.998901
15,root_cause_only,RandomForest,0.452185,0.049276,0.130000,0.194632,0.938624,0.000000,0.997669


In [25]:
# RCL: macro-F1 + ranking metrics for direct comparison with Section 4.3 tables
rcl_view = final[final['task'] == 'RCL'][[
    'variant', 'classifier',
    'macro_f1_mean', 'macro_f1_std',
    'hr_1_mean', 'hr_1_std',
    'hr_3_mean', 'hr_3_std',
    'ndcg_3_mean', 'ndcg_3_std',
]].copy()
rcl_view


,variant,classifier,macro_f1_mean,macro_f1_std,hr_1_mean,hr_1_std,hr_3_mean,hr_3_std,ndcg_3_mean,ndcg_3_std
6,full_graph,GaussianNB,0.320911,0.037877,0.860845,0.003165,0.969639,0.001640,0.917687,0.003586
7,full_graph,DecisionTree,0.292710,0.004400,0.862450,0.006722,0.968898,0.001782,0.879695,0.005887
8,full_graph,KNN,0.273298,0.009125,0.836470,0.006949,0.970133,0.000766,0.915285,0.002462
9,full_graph,RandomForest,0.285540,0.015909,0.859611,0.001872,0.970380,0.002914,0.929051,0.002470
10,full_graph,LogReg,0.278542,0.020622,0.858254,0.004512,0.969886,0.003064,0.927926,0.002673
11,full_graph,LightGBM,0.297319,0.015923,0.862203,0.003286,0.970935,0.002050,0.930280,0.001443
18,root_cause_only,GaussianNB,0.992462,0.002769,0.997038,0.000928,0.999568,0.000247,0.998665,0.000332
19,root_cause_only,DecisionTree,0.946204,0.010221,0.994199,0.001193,0.995248,0.001114,0.994825,0.000933
20,root_cause_only,KNN,0.969655,0.017174,0.995927,0.001286,0.997285,0.000924,0.996717,0.000996
21,root_cause_only,RandomForest,0.963726,0.008604,0.995125,0.001329,0.996236,0.000714,0.995697,0.000889


In [26]:
# Minority RCL per-class F1: where the imbalance really bites.
rcl_minority_cols = ['variant', 'classifier'] + [f'f1_{c}_mean' for c in RCL_MINORITY]
rcl_minority_view = final[final['task'] == 'RCL'][rcl_minority_cols].copy()
rcl_minority_view['minority_avg'] = rcl_minority_view[
    [f'f1_{c}_mean' for c in RCL_MINORITY]
].mean(axis=1)
rcl_minority_view


,variant,classifier,f1_webservice1_mean,f1_webservice2_mean,f1_dbservice1_mean,f1_dbservice2_mean,f1_logservice1_mean,f1_logservice2_mean,f1_redisservice1_mean,f1_redisservice2_mean,minority_avg
6,full_graph,GaussianNB,0.252264,0.277186,0.202848,0.169673,0.175661,0.119850,0.061053,0.168812,0.178418
7,full_graph,DecisionTree,0.174202,0.248710,0.199639,0.192126,0.065068,0.046866,0.134660,0.086762,0.143504
8,full_graph,KNN,0.153443,0.152571,0.139645,0.241482,0.092491,0.141778,0.025000,0.059893,0.125788
9,full_graph,RandomForest,0.249982,0.275196,0.182121,0.185403,0.104262,0.084950,0.000000,0.000000,0.135239
10,full_graph,LogReg,0.209974,0.139865,0.174154,0.181753,0.081961,0.090767,0.059718,0.063492,0.125211
11,full_graph,LightGBM,0.277894,0.181215,0.236886,0.236925,0.104654,0.057492,0.045000,0.059234,0.149913
18,root_cause_only,GaussianNB,0.987879,0.983129,0.987097,0.988889,1.000000,0.992593,0.990476,1.000000,0.991258
19,root_cause_only,DecisionTree,0.977699,0.987879,0.657100,0.989466,0.962393,0.919617,0.981781,0.991304,0.933405
20,root_cause_only,KNN,0.976413,0.968145,0.772122,0.994595,1.000000,1.000000,0.990476,1.000000,0.962719
21,root_cause_only,RandomForest,0.993939,0.924552,0.738573,0.994595,1.000000,1.000000,0.990476,1.000000,0.955267


## Human-readable summary

In [27]:
def format_summary(final):
    lines = []
    for variant in ['full_graph', 'root_cause_only']:
        lines.append('')
        lines.append('=' * 72)
        lines.append(f'Variant: {variant}')
        lines.append('=' * 72)
        for task in ['FTI', 'RCL']:
            sub = final[(final['variant'] == variant) & (final['task'] == task)]
            if sub.empty:
                continue
            lines.append('')
            lines.append(f'--- {task} ---')
            for _, row in sub.iterrows():
                line = (f"  {row['classifier']:>18s}: macro-F1 = "
                        f"{row.get('macro_f1_mean', float('nan')):.3f} "
                        f"\u00b1 {row.get('macro_f1_std', float('nan')):.3f}")
                if task == 'RCL' and not pd.isna(row.get('hr_1_mean', np.nan)):
                    line += (
                        f" | HR@1 = {row['hr_1_mean']:.3f}"
                        f" | HR@3 = {row['hr_3_mean']:.3f}"
                        f" | NDCG@3 = {row['ndcg_3_mean']:.3f}"
                    )
                lines.append(line)
    return '\n'.join(lines)

summary_txt = format_summary(final)
print(summary_txt)



Variant: full_graph

--- FTI ---
          GaussianNB: macro-F1 = 0.586 ± 0.071
        DecisionTree: macro-F1 = 0.728 ± 0.058
                 KNN: macro-F1 = 0.644 ± 0.052
        RandomForest: macro-F1 = 0.745 ± 0.036
              LogReg: macro-F1 = 0.708 ± 0.051
            LightGBM: macro-F1 = 0.770 ± 0.026

--- RCL ---
          GaussianNB: macro-F1 = 0.321 ± 0.038 | HR@1 = 0.861 | HR@3 = 0.970 | NDCG@3 = 0.918
        DecisionTree: macro-F1 = 0.293 ± 0.004 | HR@1 = 0.862 | HR@3 = 0.969 | NDCG@3 = 0.880
                 KNN: macro-F1 = 0.273 ± 0.009 | HR@1 = 0.836 | HR@3 = 0.970 | NDCG@3 = 0.915
        RandomForest: macro-F1 = 0.286 ± 0.016 | HR@1 = 0.860 | HR@3 = 0.970 | NDCG@3 = 0.929
              LogReg: macro-F1 = 0.279 ± 0.021 | HR@1 = 0.858 | HR@3 = 0.970 | NDCG@3 = 0.928
            LightGBM: macro-F1 = 0.297 ± 0.016 | HR@1 = 0.862 | HR@3 = 0.971 | NDCG@3 = 0.930

Variant: root_cause_only

--- FTI ---
          GaussianNB: macro-F1 = 0.452 ± 0.027
        DecisionTree:

## Save artifacts

In [28]:
csv_path = OUT_DIR / 'pilot_sklearn_gaia_results_fair.csv'
final.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

summary_path = OUT_DIR / 'pilot_sklearn_gaia_summary_fair.txt'
summary_path.write_text(summary_txt)
print(f'Saved: {summary_path}')


Saved: ../figures/pilot_sklearn_gaia_results_fair.csv
Saved: ../figures/pilot_sklearn_gaia_summary_fair.txt


In [29]:
fti_pc = fti_view.copy()
fti_pc.insert(0, 'task', 'FTI')
rcl_pc = rcl_minority_view.copy()
rcl_pc.insert(0, 'task', 'RCL')
per_class = pd.concat([fti_pc, rcl_pc], ignore_index=True, sort=False)
per_class_path = OUT_DIR / 'pilot_sklearn_gaia_per_class_fair.csv'
per_class.to_csv(per_class_path, index=False)
print(f'Saved: {per_class_path}')


Saved: ../figures/pilot_sklearn_gaia_per_class_fair.csv


## What changed vs the previous (leaky) version

| Aspect | Old notebook | This notebook |
|--------|--------------|---------------|
| Embedding source | Pre-computed `tmp/*.pkl` (one global FastText) | Recomputed every fold |
| FastText train set | Unknown (likely original 15/85 split) | Exactly the fold's train indices |
| Test-row labels leaked into FastText? | Yes (preprocessing leakage) | No |
| Comparable to TransTVDiag CV in `main.py`? | No | **Yes** — same preprocessing pipeline |

Expected outcomes once this notebook is run on the server:

- **`root_cause_only` numbers will drop the most.** The previous 0.99+ macro-F1 on RCL was almost entirely the `__label__{node_idx}{type_idx}` marker leaking through global FastText. Per-fold embeddings keep the marker (FastText is supervised), but it only sees train labels, so test rows no longer get "free" cluster membership.
- **`full_graph` FTI numbers will drop**, probably below TransTVDiag's 0.723 CE baseline. Section 4.1.3's intended message — *simple ML undershoots Graph Transformer, motivating the architecture* — is back.
- **`full_graph` RCL HR@1** will likely stay ~0.86 across all classifiers; this is the majority-class (mobservice1/2) anchor and is not affected by leakage.

## Next steps for the thesis text

1. Run this notebook on the server.
2. Compare `pilot_sklearn_gaia_results_fair.csv` against `pilot_sklearn_gaia_results.csv` (leaky version) to quantify the leakage gap — this is itself an interesting methodological observation worth a footnote.
3. Use the fair numbers to populate the table at `thesis/chapter4_experiments.md:89-94` and the summary row at line 106.